<a href="https://colab.research.google.com/github/RajShekhar0341/DataScience/blob/main/SME_QuantumComputing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.impute import SimpleImputer
import statsmodels.api as sm

os.makedirs("output", exist_ok=True)

# -------------------------
# 1. Load dataset
# -------------------------
path = "sme_quantum_cybersecurity_survey_dataset.csv"
df = pd.read_csv(path)

# -------------------------
# 2. Define constructs
# -------------------------
constructs = {
    'BI': ['BI1', 'BI2', 'BI3'],
    'PV': ['PV1', 'PV2', 'PV3'],
    'PS': ['PS1', 'PS2', 'PS3'],
    'SE': ['SE1', 'SE2', 'SE3'],
    'RE': ['RE1', 'RE2', 'RE3'],
    'TA': ['TA1', 'TA2', 'TA3'],
    'AR': ['AR1', 'AR2', 'AR3'],
    'SN': ['SN1', 'SN2', 'SN3']
}

item_cols = sum(constructs.values(), [])

# -------------------------
# 3. Clean data
# -------------------------
for c in item_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

imputer = SimpleImputer(strategy='median')
df[item_cols] = imputer.fit_transform(df[item_cols])

# -------------------------
# 4. Composite scores
# -------------------------
for k, cols in constructs.items():
    df[f"{k}_mean"] = df[cols].mean(axis=1)
    df[f"{k}_sd_items"] = df[cols].std(axis=1)

# -------------------------
# 5. Cronbach's alpha
# -------------------------
def cronbach_alpha(dataframe):
    x = dataframe.to_numpy(dtype=float)
    n_items = x.shape[1]
    item_vars = x.var(axis=0, ddof=1)
    total_var = x.sum(axis=1).var(ddof=1)
    return (n_items / (n_items - 1)) * (1 - item_vars.sum() / total_var)

alpha_rows = []
for k, cols in constructs.items():
    alpha_rows.append({
        'Construct': k,
        'Cronbach_alpha': round(cronbach_alpha(df[cols]), 3)
    })

alpha_df = pd.DataFrame(alpha_rows)

# -------------------------
# 6. Descriptive statistics
# -------------------------
desc = df[[f"{k}_mean" for k in constructs]].agg(['mean', 'std', 'min', 'max']).T.reset_index()
desc.columns = ['Construct', 'Mean', 'Std', 'Min', 'Max']
desc['Construct'] = desc['Construct'].str.replace('_mean', '', regex=False)

# -------------------------
# 7. Correlation matrix
# -------------------------
corr = df[[f"{k}_mean" for k in constructs]].corr()

# -------------------------
# 8. Regression model
# -------------------------
X = df[[f"{k}_mean" for k in ['PV', 'PS', 'SE', 'RE', 'TA', 'AR', 'SN']]]
X = sm.add_constant(X)
y = df['BI_mean']

model = sm.OLS(y, X).fit()

regression_results = pd.DataFrame({
    'Term': model.params.index,
    'Coefficient': model.params.values,
    'P_Value': model.pvalues.values,
    'T_Value': model.tvalues.values
})

# -------------------------
# 9. Quantum readiness checklist
# -------------------------
qr_items = pd.DataFrame([
    ['Cryptographic inventory exists', 1],
    ['RSA/ECC dependencies identified', 1],
    ['Long-term sensitive data classified', 1],
    ['Hybrid PQC pilot planned', 1],
    ['MFA implemented', 2],
    ['Zero-trust principles started', 1],
    ['Vendor PQC readiness reviewed', 0],
    ['Backup encryption reviewed', 2],
    ['Incident response plan updated', 2],
    ['Security awareness training conducted', 2],
], columns=['Checklist_Item', 'Score'])

qr_items['Max_Score'] = 2
qr_total = qr_items['Score'].sum()
qr_max = qr_items['Max_Score'].sum()
qr_percent = round(100 * qr_total / qr_max, 2)

def qr_level(p):
    if p < 40:
        return 'Low'
    elif p < 70:
        return 'Moderate'
    return 'High'

qr_summary = pd.DataFrame([{
    'Total_Score': qr_total,
    'Max_Score': qr_max,
    'Readiness_Percent': qr_percent,
    'Readiness_Level': qr_level(qr_percent)
}])

qr_items['Readiness_Level'] = qr_items['Score'].map({
    0: 'Not Implemented',
    1: 'Partial',
    2: 'Full'
})

# -------------------------
# 10. Export tables
# -------------------------
alpha_df.to_csv('output/reliability_cronbach_alpha.csv', index=False)
desc.to_csv('output/descriptive_statistics.csv', index=False)
corr.to_csv('output/construct_correlation_matrix.csv')
regression_results.to_csv('output/regression_results_bi.csv', index=False)
qr_items.to_csv('output/quantum_readiness_checklist_scored.csv', index=False)
qr_summary.to_csv('output/quantum_readiness_summary.csv', index=False)

summary = desc.merge(alpha_df, on='Construct', how='left')
summary.to_csv('output/summary_construct_stats.csv', index=False)

# -------------------------
# 11. Plot: construct means and reliability
# -------------------------
fig, ax1 = plt.subplots(figsize=(11, 6))

ax1.bar(summary['Construct'], summary['Mean'], color='#4C78A8', alpha=0.85)
ax1.set_ylabel('Mean score', fontsize=12)
ax1.set_xlabel('Construct', fontsize=12)
ax1.set_title('Construct Means and Reliability', fontsize=14)
ax1.set_ylim(0, 5)
ax1.tick_params(axis='x', rotation=0)

ax2 = ax1.twinx()
ax2.plot(summary['Construct'], summary['Cronbach_alpha'], color='#F58518', marker='o', linewidth=2)
ax2.set_ylabel('Cronbach alpha', fontsize=12)
ax2.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('output/construct_means_alpha.png', dpi=300, bbox_inches='tight')
plt.close()

# -------------------------
# 12. Plot: quantum readiness checklist
# -------------------------
fig, ax = plt.subplots(figsize=(12, 6))
colors = qr_items['Score'].map({0: '#E45756', 1: '#F2CF5B', 2: '#54A24B'})
ax.barh(qr_items['Checklist_Item'], qr_items['Score'], color=colors)
ax.set_xlabel('Score (0-2)', fontsize=12)
ax.set_title(f'Quantum Readiness Checklist Score: {qr_percent}% ({qr_level(qr_percent)})', fontsize=14)
ax.set_xlim(0, 2)
plt.tight_layout()
plt.savefig('output/quantum_readiness_checklist.png', dpi=300, bbox_inches='tight')
plt.close()

# -------------------------
# 13. Save analysis notes
# -------------------------
readme = f"""
Analysis Workflow

Files created:
- reliability_cronbach_alpha.csv
- descriptive_statistics.csv
- construct_correlation_matrix.csv
- regression_results_bi.csv
- quantum_readiness_checklist_scored.csv
- quantum_readiness_summary.csv
- summary_construct_stats.csv
- construct_means_alpha.png
- quantum_readiness_checklist.png

Dataset loaded:
- {path}

Steps:
1. Load survey dataset.
2. Clean and impute missing item values using median.
3. Compute composite construct scores.
4. Compute Cronbach's alpha for each construct.
5. Generate descriptive statistics.
6. Build correlation matrix.
7. Run OLS regression predicting BI from PMT constructs.
8. Score quantum readiness checklist.
9. Export tables and charts.
"""

with open('output/analysis_workflow_readme.md', 'w') as f:
    f.write(readme)

# -------------------------
# 14. Display summary
# -------------------------
print(summary.round(3).to_string(index=False))
print("\nRegression R-squared:", round(model.rsquared, 3))
print("\nQuantum readiness summary:")
print(qr_summary.to_string(index=False))

Construct  Mean   Std   Min  Max  Cronbach_alpha
       BI 4.672 0.426 2.667  5.0           0.728
       PV 3.585 0.725 1.333  5.0           0.825
       PS 3.889 0.664 1.000  5.0           0.772
       SE 3.139 0.831 1.000  5.0           0.857
       RE 3.813 0.649 1.667  5.0           0.761
       TA 3.437 0.672 2.000  5.0           0.786
       AR 3.581 0.829 1.000  5.0           0.848
       SN 3.315 0.802 1.000  5.0           0.844

Regression R-squared: 0.188

Quantum readiness summary:
 Total_Score  Max_Score  Readiness_Percent Readiness_Level
          13         20               65.0        Moderate
